# Martian Map – extraction des données et itinéraire (mont Sharp) · version 3

Contenu : calibration de la pente sur l'inclinaison réelle de Curiosity ; couche crêtes et creux ; cibles choisies automatiquement (une crête, un creux) ; itinéraire le plus rapide et sûr (règle de Tobler) ; temps de marche ; dose de radiations (RAD) ; **`mission.json` + l'interface `app.html` à deux écrans** (planification dans le rover et mode marche).

Exécutez les cellules **dans l'ordre**. À la fin, `sortie.zip` contient tout le site : dézippez-le et ouvrez `app.html`.


## Étape 1 – Installer les bibliothèques (1 à 2 min)

In [ ]:
!pip install -q rasterio pyproj scikit-image

## Étape 2 – Créer les fichiers du projet
Les positions de Curiosity dans la zone des boxworks (sols 4534 à 4830), le script de calcul, l'interface `app.html` et une copie locale de Leaflet (pour que la page s'ouvre sans réseau).


In [ ]:
%%writefile curiosity_boxworks_waypoints.geojson
{"type": "FeatureCollection", "name": "Curiosity - zone des boxworks, sols 4534 a 4830 (source NASA/JPL MMGIS MSL_waypoints)", "features": [{"type": "Feature", "properties": {"sol": 4534, "elev_m": -3658, "tilt_deg": 9.0}, "geometry": {"type": "Point", "coordinates": [137.38644, -4.79762]}}, {"type": "Feature", "properties": {"sol": 4537, "elev_m": -3656, "tilt_deg": 9.8}, "geometry": {"type": "Point", "coordinates": [137.3862, -4.79779]}}, {"type": "Feature", "properties": {"sol": 4539, "elev_m": -3655, "tilt_deg": 7.0}, "geometry": {"type": "Point", "coordinates": [137.38592, -4.79782]}}, {"type": "Feature", "properties": {"sol": 4541, "elev_m": -3655, "tilt_deg": 6.0}, "geometry": {"type": "Point", "coordinates": [137.38593, -4.79782]}}, {"type": "Feature", "properties": {"sol": 4544, "elev_m": -3654, "tilt_deg": 2.3}, "geometry": {"type": "Point", "coordinates": [137.38566, -4.79796]}}, {"type": "Feature", "properties": {"sol": 4546, "elev_m": -3653, "tilt_deg": 10.0}, "geometry": {"type": "Point", "coordinates": [137.38501, -4.79827]}}, {"type": "Feature", "properties": {"sol": 4548, "elev_m": -3651, "tilt_deg": 6.6}, "geometry": {"type": "Point", "coordinates": [137.38405, -4.79879]}}, {"type": "Feature", "properties": {"sol": 4552, "elev_m": -3648, "tilt_deg": 7.2}, "geometry": {"type": "Point", "coordinates": [137.38357, -4.79927]}}, {"type": "Feature", "properties": {"sol": 4554, "elev_m": -3646, "tilt_deg": 8.4}, "geometry": {"type": "Point", "coordinates": [137.38344, -4.79971]}}, {"type": "Feature", "properties": {"sol": 4558, "elev_m": -3647, "tilt_deg": 5.8}, "geometry": {"type": "Point", "coordinates": [137.3835, -4.79981]}}, {"type": "Feature", "properties": {"sol": 4560, "elev_m": -3647, "tilt_deg": 8.3}, "geometry": {"type": "Point", "coordinates": [137.38354, -4.79985]}}, {"type": "Feature", "properties": {"sol": 4571, "elev_m": -3644, "tilt_deg": 2.2}, "geometry": {"type": "Point", "coordinates": [137.38299, -4.80041]}}, {"type": "Feature", "properties": {"sol": 4573, "elev_m": -3644, "tilt_deg": 4.5}, "geometry": {"type": "Point", "coordinates": [137.38252, -4.80116]}}, {"type": "Feature", "properties": {"sol": 4575, "elev_m": -3644, "tilt_deg": 4.8}, "geometry": {"type": "Point", "coordinates": [137.38211, -4.80191]}}, {"type": "Feature", "properties": {"sol": 4578, "elev_m": -3645, "tilt_deg": 5.3}, "geometry": {"type": "Point", "coordinates": [137.38171, -4.80193]}}, {"type": "Feature", "properties": {"sol": 4580, "elev_m": -3644, "tilt_deg": 3.5}, "geometry": {"type": "Point", "coordinates": [137.38125, -4.80233]}}, {"type": "Feature", "properties": {"sol": 4582, "elev_m": -3644, "tilt_deg": 3.2}, "geometry": {"type": "Point", "coordinates": [137.38125, -4.80235]}}, {"type": "Feature", "properties": {"sol": 4585, "elev_m": -3644, "tilt_deg": 4.4}, "geometry": {"type": "Point", "coordinates": [137.38105, -4.80257]}}, {"type": "Feature", "properties": {"sol": 4587, "elev_m": -3643, "tilt_deg": 7.9}, "geometry": {"type": "Point", "coordinates": [137.38079, -4.80274]}}, {"type": "Feature", "properties": {"sol": 4588, "elev_m": -3645, "tilt_deg": 7.0}, "geometry": {"type": "Point", "coordinates": [137.3806, -4.80243]}}, {"type": "Feature", "properties": {"sol": 4591, "elev_m": -3645, "tilt_deg": 13.9}, "geometry": {"type": "Point", "coordinates": [137.38045, -4.80251]}}, {"type": "Feature", "properties": {"sol": 4594, "elev_m": -3642, "tilt_deg": 6.9}, "geometry": {"type": "Point", "coordinates": [137.38043, -4.80296]}}, {"type": "Feature", "properties": {"sol": 4596, "elev_m": -3640, "tilt_deg": 6.5}, "geometry": {"type": "Point", "coordinates": [137.38007, -4.80361]}}, {"type": "Feature", "properties": {"sol": 4598, "elev_m": -3636, "tilt_deg": 9.2}, "geometry": {"type": "Point", "coordinates": [137.37976, -4.80402]}}, {"type": "Feature", "properties": {"sol": 4600, "elev_m": -3633, "tilt_deg": 17.0}, "geometry": {"type": "Point", "coordinates": [137.37969, -4.80419]}}, {"type": "Feature", "properties": {"sol": 4602, "elev_m": -3631, "tilt_deg": 6.8}, "geometry": {"type": "Point", "coordinates": [137.37962, -4.80437]}}, {"type": "Feature", "properties": {"sol": 4605, "elev_m": -3631, "tilt_deg": 4.0}, "geometry": {"type": "Point", "coordinates": [137.37956, -4.80443]}}, {"type": "Feature", "properties": {"sol": 4609, "elev_m": -3624, "tilt_deg": 13.4}, "geometry": {"type": "Point", "coordinates": [137.37932, -4.80474]}}, {"type": "Feature", "properties": {"sol": 4612, "elev_m": -3624, "tilt_deg": 10.0}, "geometry": {"type": "Point", "coordinates": [137.3793, -4.80495]}}, {"type": "Feature", "properties": {"sol": 4614, "elev_m": -3622, "tilt_deg": 6.6}, "geometry": {"type": "Point", "coordinates": [137.37945, -4.80521]}}, {"type": "Feature", "properties": {"sol": 4616, "elev_m": -3621, "tilt_deg": 7.4}, "geometry": {"type": "Point", "coordinates": [137.37952, -4.80545]}}, {"type": "Feature", "properties": {"sol": 4618, "elev_m": -3620, "tilt_deg": 9.8}, "geometry": {"type": "Point", "coordinates": [137.37954, -4.8055]}}, {"type": "Feature", "properties": {"sol": 4621, "elev_m": -3620, "tilt_deg": 4.8}, "geometry": {"type": "Point", "coordinates": [137.37951, -4.80541]}}, {"type": "Feature", "properties": {"sol": 4623, "elev_m": -3622, "tilt_deg": 7.4}, "geometry": {"type": "Point", "coordinates": [137.37992, -4.80548]}}, {"type": "Feature", "properties": {"sol": 4625, "elev_m": -3618, "tilt_deg": 9.8}, "geometry": {"type": "Point", "coordinates": [137.37976, -4.80586]}}, {"type": "Feature", "properties": {"sol": 4628, "elev_m": -3614, "tilt_deg": 11.8}, "geometry": {"type": "Point", "coordinates": [137.37946, -4.80599]}}, {"type": "Feature", "properties": {"sol": 4629, "elev_m": -3614, "tilt_deg": 12.1}, "geometry": {"type": "Point", "coordinates": [137.37972, -4.80617]}}, {"type": "Feature", "properties": {"sol": 4633, "elev_m": -3611, "tilt_deg": 13.6}, "geometry": {"type": "Point", "coordinates": [137.37967, -4.80646]}}, {"type": "Feature", "properties": {"sol": 4634, "elev_m": -3610, "tilt_deg": 17.5}, "geometry": {"type": "Point", "coordinates": [137.37912, -4.80636]}}, {"type": "Feature", "properties": {"sol": 4636, "elev_m": -3607, "tilt_deg": 13.3}, "geometry": {"type": "Point", "coordinates": [137.37934, -4.80654]}}, {"type": "Feature", "properties": {"sol": 4639, "elev_m": -3607, "tilt_deg": 15.0}, "geometry": {"type": "Point", "coordinates": [137.37932, -4.80655]}}, {"type": "Feature", "properties": {"sol": 4641, "elev_m": -3608, "tilt_deg": 13.3}, "geometry": {"type": "Point", "coordinates": [137.37935, -4.80669]}}, {"type": "Feature", "properties": {"sol": 4643, "elev_m": -3605, "tilt_deg": 13.2}, "geometry": {"type": "Point", "coordinates": [137.37941, -4.8068]}}, {"type": "Feature", "properties": {"sol": 4647, "elev_m": -3610, "tilt_deg": 8.3}, "geometry": {"type": "Point", "coordinates": [137.37981, -4.80689]}}, {"type": "Feature", "properties": {"sol": 4650, "elev_m": -3611, "tilt_deg": 11.9}, "geometry": {"type": "Point", "coordinates": [137.37984, -4.80689]}}, {"type": "Feature", "properties": {"sol": 4653, "elev_m": -3608, "tilt_deg": 10.0}, "geometry": {"type": "Point", "coordinates": [137.3799, -4.80708]}}, {"type": "Feature", "properties": {"sol": 4655, "elev_m": -3608, "tilt_deg": 4.7}, "geometry": {"type": "Point", "coordinates": [137.3799, -4.80717]}}, {"type": "Feature", "properties": {"sol": 4660, "elev_m": -3605, "tilt_deg": 8.2}, "geometry": {"type": "Point", "coordinates": [137.38006, -4.80745]}}, {"type": "Feature", "properties": {"sol": 4662, "elev_m": -3606, "tilt_deg": 8.6}, "geometry": {"type": "Point", "coordinates": [137.38019, -4.80763]}}, {"type": "Feature", "properties": {"sol": 4664, "elev_m": -3605, "tilt_deg": 5.6}, "geometry": {"type": "Point", "coordinates": [137.38025, -4.80793]}}, {"type": "Feature", "properties": {"sol": 4667, "elev_m": -3605, "tilt_deg": 8.2}, "geometry": {"type": "Point", "coordinates": [137.38051, -4.80802]}}, {"type": "Feature", "properties": {"sol": 4669, "elev_m": -3607, "tilt_deg": 7.4}, "geometry": {"type": "Point", "coordinates": [137.38088, -4.80805]}}, {"type": "Feature", "properties": {"sol": 4671, "elev_m": -3608, "tilt_deg": 8.3}, "geometry": {"type": "Point", "coordinates": [137.38127, -4.80827]}}, {"type": "Feature", "properties": {"sol": 4673, "elev_m": -3607, "tilt_deg": 8.6}, "geometry": {"type": "Point", "coordinates": [137.38157, -4.80872]}}, {"type": "Feature", "properties": {"sol": 4677, "elev_m": -3607, "tilt_deg": 4.4}, "geometry": {"type": "Point", "coordinates": [137.38169, -4.80871]}}, {"type": "Feature", "properties": {"sol": 4680, "elev_m": -3608, "tilt_deg": 8.9}, "geometry": {"type": "Point", "coordinates": [137.38125, -4.80827]}}, {"type": "Feature", "properties": {"sol": 4684, "elev_m": -3608, "tilt_deg": 9.3}, "geometry": {"type": "Point", "coordinates": [137.38102, -4.80801]}}, {"type": "Feature", "properties": {"sol": 4687, "elev_m": -3610, "tilt_deg": 6.7}, "geometry": {"type": "Point", "coordinates": [137.38104, -4.80784]}}, {"type": "Feature", "properties": {"sol": 4709, "elev_m": -3609, "tilt_deg": 9.7}, "geometry": {"type": "Point", "coordinates": [137.38105, -4.808]}}, {"type": "Feature", "properties": {"sol": 4711, "elev_m": -3608, "tilt_deg": 11.0}, "geometry": {"type": "Point", "coordinates": [137.381, -4.80801]}}, {"type": "Feature", "properties": {"sol": 4741, "elev_m": -3612, "tilt_deg": 12.2}, "geometry": {"type": "Point", "coordinates": [137.38101, -4.80776]}}, {"type": "Feature", "properties": {"sol": 4748, "elev_m": -3610, "tilt_deg": 14.1}, "geometry": {"type": "Point", "coordinates": [137.38112, -4.80791]}}, {"type": "Feature", "properties": {"sol": 4752, "elev_m": -3608, "tilt_deg": 9.5}, "geometry": {"type": "Point", "coordinates": [137.38099, -4.80803]}}, {"type": "Feature", "properties": {"sol": 4789, "elev_m": -3608, "tilt_deg": 11.5}, "geometry": {"type": "Point", "coordinates": [137.38099, -4.80801]}}, {"type": "Feature", "properties": {"sol": 4805, "elev_m": -3611, "tilt_deg": 4.4}, "geometry": {"type": "Point", "coordinates": [137.38142, -4.80784]}}, {"type": "Feature", "properties": {"sol": 4810, "elev_m": -3613, "tilt_deg": 7.2}, "geometry": {"type": "Point", "coordinates": [137.3815, -4.80754]}}, {"type": "Feature", "properties": {"sol": 4813, "elev_m": -3616, "tilt_deg": 11.5}, "geometry": {"type": "Point", "coordinates": [137.38167, -4.80735]}}, {"type": "Feature", "properties": {"sol": 4816, "elev_m": -3615, "tilt_deg": 6.7}, "geometry": {"type": "Point", "coordinates": [137.38194, -4.80753]}}, {"type": "Feature", "properties": {"sol": 4818, "elev_m": -3616, "tilt_deg": 4.8}, "geometry": {"type": "Point", "coordinates": [137.38256, -4.8079]}}, {"type": "Feature", "properties": {"sol": 4820, "elev_m": -3617, "tilt_deg": 3.7}, "geometry": {"type": "Point", "coordinates": [137.38267, -4.80778]}}, {"type": "Feature", "properties": {"sol": 4823, "elev_m": -3616, "tilt_deg": 7.6}, "geometry": {"type": "Point", "coordinates": [137.38263, -4.80788]}}, {"type": "Feature", "properties": {"sol": 4827, "elev_m": -3615, "tilt_deg": 9.1}, "geometry": {"type": "Point", "coordinates": [137.38285, -4.80813]}}, {"type": "Feature", "properties": {"sol": 4830, "elev_m": -3609, "tilt_deg": 10.0}, "geometry": {"type": "Point", "coordinates": [137.38265, -4.80882]}}]}

In [ ]:
%%writefile martian_map_pipeline.py
"""
Martian Map – pipeline de préparation des couches et de l'itinéraire (mont Sharp, cratère Gale)

Ce que fait ce script :
  1. définit une petite zone (par défaut 3 x 3 km) autour d'un centre lon/lat ;
  2. découpe UNIQUEMENT cette zone dans le modèle de terrain 1 m (DEM) et l'orthomosaïque
     couleur HiRISE de Gale (USGS), en lecture distante, sans télécharger les fichiers entiers ;
  3. calcule les couches dérivées : ombrage du relief, pentes, rugosité, carte des dangers ;
  4. calcule l'itinéraire le moins risqué entre l'habitat et les cibles (chemin de moindre coût) ;
  5. exporte les couches (PNG), l'itineraire (GeoJSON) et mission.json, lu par app.html.

Installation :
  pip install rasterio pyproj numpy scipy scikit-image pillow

Conseil : lancez-le dans Google Colab si le réseau sur place est lent.
Les seuils de pente sont des HYPOTHÈSES de travail : ajustez-les et justifiez-les au pitch.
"""

import json
import math
import os
import shutil

import numpy as np
import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from pyproj import Transformer, CRS
from scipy import ndimage
from skimage.graph import route_through_array
from PIL import Image

# ----------------------------------------------------------------------------
# 1. PARAMÈTRES À REMPLIR PAR L'ÉQUIPE
# ----------------------------------------------------------------------------

# Données USGS (MSL Gale). Vérifiez les URL sur astrogeology.usgs.gov si une lecture échoue.
DEM_URL = "/vsicurl/https://planetarymaps.usgs.gov/mosaic/Mars/MSL/MSL_Gale_DEM_Mosaic_1m_v3.tif"
ORTHO_URL = "/vsicurl/https://asc-pds-services.s3.us-west-2.amazonaws.com/mosaic/Mars/MSL/MSL_Gale_HiRISE-LRGB_78quads_sharp_cog.tif"

# Positions de Curiosity exportées en GeoJSON (carte « Where is Curiosity », couche Waypoints).
# Laissez None pour fixer le centre à la main (CENTER_LON / CENTER_LAT).
WAYPOINTS_GEOJSON = "curiosity_boxworks_waypoints.geojson"   # fourni avec ce script
SOL_MIN, SOL_MAX = 4534, 4830   # traversée des boxworks par Curiosity (mai 2025 -> début 2026)

CENTER_LON = 137.38129   # centre de la traversée (calculé sur les 73 positions)
CENTER_LAT = -4.80491
ZONE_KM = 2.0            # carré de 2 x 2 km : couvre toute la traversée avec de la marge

# Points de la mission (lon, lat), pris sur les positions réelles de Curiosity.
HABITAT = (137.38566, -4.79796)   # sol 4544 : rover presque à plat (inclinaison 2,3°)
CIBLES = [
    (137.37941, -4.80680),        # cible A, sol 4643 : point le plus haut de la traversée
    (137.38101, -4.80800),        # cible B, sols 4684-4789 : site où Curiosity est resté le plus longtemps
]

# Hypothèses de sécurité (à justifier) : pente en degrés
PENTE_PRUDENCE = 10.0
PENTE_INTERDITE = 20.0
RUGOSITE_INTERDITE = 1.5   # écart-type local d'altitude (m) sur 5 x 5 m
PAS_ITINERAIRE_M = 4       # rééchantillonnage pour accélérer le calcul (1 m -> 4 m)
LISSAGES_M = [0, 1, 2, 3, 5]   # échelles testées pour calibrer la pente sur l'inclinaison réelle de Curiosity

# Couche « crêtes et creux » : indice de position topographique (altitude - moyenne du voisinage)
FENETRE_TPI_M = 21           # taille du voisinage comparé (m)
CIBLES_AUTO = True           # cherche la crête la plus nette autour de A et le creux le plus net autour de B
RAYON_RECHERCHE_M = 120      # rayon de recherche autour des points de départ

# Temps de marche : règle de Tobler (vitesse d'un randonneur selon la pente), puis scénario prudent
FACTEUR_VITESSE_PRUDENT = 0.5   # HYPOTHÈSE : combinaison + charge divisent la vitesse par 2 (à justifier)
DOSE_RAD_MSV_JOUR = 0.67        # dose moyenne mesurée par RAD (Curiosity, Gale) en surface, 2012-2013
DUREE_SOL_H = 24.66             # durée d'un jour martien
DUREE_ARRET_MIN = 20            # arrêt scientifique prévu sur chaque cible (minutes)
OPTIQUE_WEB_PX = 4096           # largeur max de l'image optique servie au navigateur (GitHub Pages)

# Mesures d'ambiance affichées par le panneau « Conditions » de app.html.
# Ce sont les dernières valeurs PUBLIÉES par les instruments de Curiosity : lors d'une vraie
# mission, cette liste serait alimentée en direct par la station météo du rover.
CONDITIONS = {
    "maj": "Dernières mesures publiées de Curiosity – un équipage lirait ici la station météo du rover.",
    "mesures": [
        {"nom": "Radiations en surface", "valeur": f"{DOSE_RAD_MSV_JOUR} mSv/jour".replace('.', ','),
         "detail": "RAD, moyenne 2012–2013 (Hassler et al., Science 2013)", "etat": "ok"},
        {"nom": "Température de l'air", "valeur": "−70 à −10 °C",
         "detail": "REMS, cycle jour/nuit typique au cratère Gale", "etat": "ok"},
        {"nom": "Pression", "valeur": "≈ 750 Pa",
         "detail": "REMS, variation saisonnière 700 à 900 Pa", "etat": "ok"},
        {"nom": "Vent", "valeur": "2 à 10 m/s", "detail": "REMS, vents de pente diurnes", "etat": "ok"},
        {"nom": "Opacité de poussière", "valeur": "τ ≈ 0,3 à 0,6",
         "detail": "Ciel clair à voilé (Mastcam / REMS)", "etat": "ok"},
        {"nom": "Sismicité", "valeur": "pas de mesure active",
         "detail": "InSight arrêté fin 2022 – aucune surveillance sismique à Gale", "etat": "info"},
    ],
}
SOURCES = [
    "Modèle de terrain 1 m : USGS Astrogeology, MSL Gale DEM Mosaic 1 m v3",
    "Orthomosaïque HiRISE LRGB : USGS / NASA / JPL-Caltech / Univ. of Arizona",
    "Positions et inclinaisons de Curiosity : NASA/JPL MMGIS, MSL_waypoints",
    "Radiations : MSL/RAD (Hassler et al., Science 2013)",
    "Météo : MSL/REMS (Centro de Astrobiología CSIC-INTA / NASA JPL)",
]
SCIENCE_CIBLES = [
    "Argiles sur les crêtes des boxworks (Curiosity, sol 4643)",
    "Carbonates dans les creux des boxworks (Curiosity, sols 4684–4789)",
]

SORTIE = "sortie"
RAYON_MARS = 3396190.0

# ----------------------------------------------------------------------------
# 2. OUTILS
# ----------------------------------------------------------------------------

LONLAT_MARS = CRS.from_proj4(f"+proj=longlat +a={RAYON_MARS} +b={RAYON_MARS} +no_defs")


def centre_depuis_waypoints(path, sol_min, sol_max):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    pts = []
    for feat in data.get("features", []):
        props = feat.get("properties", {})
        sol = props.get("sol")
        try:
            sol = int(sol)
        except (TypeError, ValueError):
            continue
        if sol_min <= sol <= sol_max and feat.get("geometry", {}).get("type") == "Point":
            lon, lat = feat["geometry"]["coordinates"][:2]
            pts.append((lon, lat))
    if not pts:
        raise ValueError("Aucun waypoint dans l'intervalle de sols : élargissez SOL_MIN/SOL_MAX.")
    lons, lats = zip(*pts)
    return float(np.mean(lons)), float(np.mean(lats)), pts


def bbox_lonlat(lon, lat, cote_km):
    demi = cote_km * 1000 / 2
    dlat = math.degrees(demi / RAYON_MARS)
    dlon = math.degrees(demi / (RAYON_MARS * math.cos(math.radians(lat))))
    return lon - dlon, lat - dlat, lon + dlon, lat + dlat


def decouper(url, bbox, out_tif, out_shape=None):
    with rasterio.open(url) as src:
        tr = Transformer.from_crs(LONLAT_MARS, src.crs, always_xy=True)
        xs, ys = tr.transform([bbox[0], bbox[2], bbox[0], bbox[2]], [bbox[1], bbox[1], bbox[3], bbox[3]])
        win = from_bounds(min(xs), min(ys), max(xs), max(ys), src.transform)
        kw = {}
        if out_shape:
            kw = {"out_shape": (src.count, *out_shape), "resampling": Resampling.bilinear}
        data = src.read(window=win, boundless=False, **kw)
        transform = src.window_transform(win)
        if out_shape:
            transform = transform * transform.scale(win.width / out_shape[1], win.height / out_shape[0])
        profile = src.profile.copy()
        profile.update(height=data.shape[1], width=data.shape[2], transform=transform, driver="GTiff",
                       tiled=False, compress="deflate")
        profile.pop("blockxsize", None)
        profile.pop("blockysize", None)
        profile.pop("photometric", None)   # évite l'erreur YCbCr de l'orthomosaïque couleur
        with rasterio.open(out_tif, "w", **profile) as dst:
            dst.write(data)
        return data, transform, src.crs, src.nodata


def to_png(arr, path, cmap):
    """arr : float 0..1 (NaN = transparent) ; cmap : fonction -> (r, g, b)."""
    h, w = arr.shape
    rgba = np.zeros((h, w, 4), dtype=np.uint8)
    ok = ~np.isnan(arr)
    r, g, b = cmap(np.clip(np.nan_to_num(arr), 0, 1))
    rgba[..., 0], rgba[..., 1], rgba[..., 2] = r, g, b
    rgba[..., 3] = np.where(ok, 255, 0)
    Image.fromarray(rgba).save(path)


def cmap_gris(x):
    v = (x * 255).astype(np.uint8)
    return v, v, v


def cmap_pente(x):
    # vert -> jaune -> rouge
    r = np.clip(2 * x, 0, 1) * 255
    g = np.clip(2 * (1 - x), 0, 1) * 255
    return r.astype(np.uint8), g.astype(np.uint8), np.full_like(r, 40, dtype=np.uint8)


# ----------------------------------------------------------------------------
# 3. PIPELINE
# ----------------------------------------------------------------------------

def main():
    os.makedirs(SORTIE, exist_ok=True)

    # Zone
    if WAYPOINTS_GEOJSON and os.path.exists(WAYPOINTS_GEOJSON):
        lon0, lat0, wpts = centre_depuis_waypoints(WAYPOINTS_GEOJSON, SOL_MIN, SOL_MAX)
    elif CENTER_LON is not None and CENTER_LAT is not None:
        lon0, lat0, wpts = CENTER_LON, CENTER_LAT, []
    else:
        raise SystemExit("Renseignez WAYPOINTS_GEOJSON ou CENTER_LON / CENTER_LAT.")
    bbox = bbox_lonlat(lon0, lat0, ZONE_KM)
    print(f"Zone centrée sur lon {lon0:.4f}, lat {lat0:.4f} ; bbox {bbox}")

    # Couche topographique (DEM 1 m)
    dem, dem_tr, dem_crs, nodata = decouper(DEM_URL, bbox, f"{SORTIE}/dem.tif")
    z = dem[0].astype("float32")
    if nodata is not None:
        z[z == nodata] = np.nan
    px = abs(dem_tr.a)
    print(f"DEM découpé : {z.shape}, pixel ~{px:.2f} m")

    # Couche optique (orthomosaïque couleur) au même cadrage
    try:
        ortho, _, _, _ = decouper(ORTHO_URL, bbox, f"{SORTIE}/ortho.tif")
        bandes = (ortho[:3] if ortho.shape[0] >= 3 else np.repeat(ortho[:1], 3, axis=0)).astype("float32")
        lo, hi = np.percentile(bandes[bandes > 0], [1, 99])
        img = np.clip((bandes - lo) / max(hi - lo, 1e-6) * 255, 0, 255).astype("uint8")
        im_opt = Image.fromarray(np.moveaxis(img, 0, -1))
        im_opt.save(f"{SORTIE}/optique.png")
        # version web : optique.png fait souvent > 100 Mo, ce que GitHub refuse et
        # ce qu'un navigateur met longtemps à décoder. app.html utilise ce JPEG.
        if im_opt.width > OPTIQUE_WEB_PX:
            h_web = round(im_opt.height * OPTIQUE_WEB_PX / im_opt.width)
            im_opt = im_opt.resize((OPTIQUE_WEB_PX, h_web), Image.LANCZOS)
        im_opt.save(f"{SORTIE}/optique.jpg", quality=85, optimize=True, progressive=True)
        print(f"Image optique : {im_opt.width} x {im_opt.height} px pour le web (optique.jpg)")
    except Exception as e:  # noqa: BLE001
        print("Orthomosaïque non lue :", e, "(gardez optique.png obtenu par la cellule de secours)")

    zf = np.where(np.isnan(z), np.nanmean(z), z)
    mask = np.isnan(z)
    tr_inv = Transformer.from_crs(LONLAT_MARS, dem_crs, always_xy=True)
    tr_fwd = Transformer.from_crs(dem_crs, LONLAT_MARS, always_xy=True)

    def pixel(lon, lat):
        x, y = tr_inv.transform(lon, lat)
        col, row = ~dem_tr * (x, y)
        return int(row), int(col)

    # Positions et inclinaisons réelles de Curiosity (vérité terrain)
    verite = []
    if WAYPOINTS_GEOJSON and os.path.exists(WAYPOINTS_GEOJSON):
        with open(WAYPOINTS_GEOJSON, encoding="utf-8") as fh:
            for ft in json.load(fh)["features"]:
                tilt = ft["properties"].get("tilt_deg")
                r, c = pixel(*ft["geometry"]["coordinates"][:2])
                if tilt is not None and 0 <= r < z.shape[0] and 0 <= c < z.shape[1]:
                    verite.append((r, c, float(tilt)))

    def calcul_pente(sigma_m):
        zs = ndimage.gaussian_filter(zf, sigma_m / px) if sigma_m > 0 else zf
        gy, gx = np.gradient(zs, px)
        return np.degrees(np.arctan(np.hypot(gx, gy))), gx, gy

    # Calibration : à quelle échelle la pente calculée colle-t-elle au terrain vécu par Curiosity ?
    meilleur = 2
    if verite:
        rr, cc, tt = map(np.array, zip(*verite))
        print("Calibration de l'échelle de pente sur les inclinaisons mesurées par Curiosity :")
        scores = {}
        for s in LISSAGES_M:
            p, _, _ = calcul_pente(s)
            scores[s] = float(np.abs(p[rr, cc] - tt).mean())
            print(f"  lissage {s} m -> écart moyen {scores[s]:.1f}°")
        meilleur = min(scores, key=scores.get)
    pente, gx, gy = calcul_pente(meilleur)
    if verite:
        print(f"Validation (lissage retenu {meilleur} m) sur {len(verite)} positions : pente calculée moyenne "
              f"{pente[rr, cc].mean():.1f}°, inclinaison du rover {tt.mean():.1f}°, écart moyen "
              f"{np.abs(pente[rr, cc] - tt).mean():.1f}°")

    # Rugosité (blocs, crêtes) et seuil calibré : au moins ce que Curiosity a franchi
    moy = ndimage.uniform_filter(zf, 5)
    rug = np.sqrt(np.maximum(ndimage.uniform_filter(zf ** 2, 5) - moy ** 2, 0))
    seuil_rug = RUGOSITE_INTERDITE
    if verite:
        seuil_rug = max(RUGOSITE_INTERDITE, float(rug[rr, cc].max()) * 1.1)
        print(f"Rugosité maximale franchie par Curiosity : {rug[rr, cc].max():.2f} m -> seuil interdit {seuil_rug:.2f} m")

    pent_hs = np.arctan(np.hypot(gx, gy))
    az, alt = math.radians(315), math.radians(45)
    hill = np.sin(alt) * np.cos(pent_hs) + np.cos(alt) * np.sin(pent_hs) * np.cos(az - np.arctan2(-gx, gy))

    danger = np.zeros_like(pente)
    danger[pente >= PENTE_PRUDENCE] = 1
    danger[(pente >= PENTE_INTERDITE) | (rug >= seuil_rug)] = 2
    print(f"Part de la zone : facile {np.mean(danger == 0):.0%}, prudence {np.mean(danger == 1):.0%}, "
          f"interdite {np.mean(danger == 2):.0%}")

    to_png(np.where(mask, np.nan, (hill + 1) / 2), f"{SORTIE}/relief.png", cmap_gris)
    to_png(np.where(mask, np.nan, np.clip(pente / 30, 0, 1)), f"{SORTIE}/pentes.png", cmap_pente)
    to_png(np.where(mask, np.nan, danger / 2), f"{SORTIE}/dangers.png", cmap_pente)

    # Couche crêtes / creux (indice de position topographique, TPI)
    zs = ndimage.gaussian_filter(zf, meilleur / px) if meilleur > 0 else zf
    w = max(3, int(round(FENETRE_TPI_M / px)) | 1)
    tpi = zs - ndimage.uniform_filter(zs, w)
    lim = float(np.nanpercentile(np.abs(tpi), 98)) or 1.0
    a = np.clip(np.abs(tpi) / lim, 0, 1)
    rgba = np.zeros((*tpi.shape, 4), dtype=np.uint8)
    rgba[..., 0] = np.where(tpi > 0, 229, 70)
    rgba[..., 1] = np.where(tpi > 0, 120, 140)
    rgba[..., 2] = np.where(tpi > 0, 50, 230)
    rgba[..., 3] = np.where(mask, 0, (a * 220).astype(np.uint8))
    Image.fromarray(rgba).save(f"{SORTIE}/cretes_creux.png")
    print(f"Couche crêtes/creux : relief local de -{lim:.1f} m (creux) à +{lim:.1f} m (crêtes)")

    def vers_lonlat_px(r, c):
        x, y = dem_tr * (c + 0.5, r + 0.5)
        return tr_fwd.transform(x, y)

    # Cibles géologiques : la crête la plus nette autour de A, le creux le plus net autour de B
    cibles = list(CIBLES)
    noms_cibles = [f"Cible {chr(65 + i)}" for i in range(len(cibles))]
    if CIBLES_AUTO and len(cibles) >= 2:
        R = int(RAYON_RECHERCHE_M / px)
        for i, (signe, nature) in enumerate([(1, "crête"), (-1, "creux")]):
            r0, c0 = pixel(*cibles[i])
            r1, r2, c1, c2 = max(r0 - R, 0), min(r0 + R, z.shape[0]), max(c0 - R, 0), min(c0 + R, z.shape[1])
            score = signe * tpi[r1:r2, c1:c2]
            score = np.where((danger[r1:r2, c1:c2] < 2) & ~mask[r1:r2, c1:c2], score, -np.inf)
            rr_, cc_ = np.unravel_index(np.argmax(score), score.shape)
            rn, cn = r1 + rr_, c1 + cc_
            cibles[i] = vers_lonlat_px(rn, cn)
            noms_cibles[i] = f"Cible {chr(65 + i)} – {nature}"
            print(f"{noms_cibles[i]} : lon {cibles[i][0]:.5f}, lat {cibles[i][1]:.5f}, relief local "
                  f"{tpi[rn, cn]:+.2f} m, à {math.hypot(rn - r0, cn - c0) * px:.0f} m du point de départ")

    # Itinéraire le plus rapide et sûr : Dijkstra sur une grille de TEMPS de marche (règle de Tobler)
    itineraire, points, stats = [], {"habitat": HABITAT, "cibles": cibles, "noms": noms_cibles}, {}
    if HABITAT and cibles:
        f = max(1, int(round(PAS_ITINERAIRE_M / px)))
        v_ms = 6.0 * np.exp(-3.5 * (np.tan(np.radians(pente)) + 0.05)) / 3.6   # Tobler, en m/s
        cout = (f * px) / v_ms                                              # secondes pour traverser une case
        cout[danger == 2] = 1e6
        cout[mask] = 1e6
        cout_r = cout[::f, ::f]
        autorise = cout_r < 1e6
        _, (ir, ic) = ndimage.distance_transform_edt(~autorise, return_indices=True)

        def cellule(lon, lat, nom):
            r, c = pixel(lon, lat)
            r, c = r // f, c // f
            r2, c2 = int(ir[r, c]), int(ic[r, c])
            if (r2, c2) != (r, c):
                print(f"{nom} était en zone interdite : déplacé de {math.hypot(r2 - r, c2 - c) * f * px:.0f} m")
            return r2, c2

        def vers_lonlat(r, c):
            x, y = dem_tr * (c * f + f / 2, r * f + f / 2)
            return tr_fwd.transform(x, y)

        cells = [cellule(*p, n) for p, n in zip([HABITAT, *cibles], ["Habitat", *noms_cibles])]
        points["habitat"] = vers_lonlat(*cells[0])
        points["cibles"] = [vers_lonlat(*cc_) for cc_ in cells[1:]]
        boucle = cells + [cells[0]]

        # une seule liste de cases, sans doublon aux jonctions : bornes[k] = index d'arrivée de l'étape k
        bornes, cases = [0], []
        for a_, b_ in zip(boucle[:-1], boucle[1:]):
            chemin, _ = route_through_array(cout_r, a_, b_, fully_connected=True, geometric=True)
            chemin = list(chemin)
            if cases and chemin and chemin[0] == cases[-1]:
                chemin = chemin[1:]
            cases += chemin
            bornes.append(len(cases) - 1)

        itineraire = [vers_lonlat(r, c) for r, c in cases]
        pentes_route = [float(pente[r * f, c * f]) for r, c in cases]
        alt_route = [float(zf[r * f, c * f]) for r, c in cases]

        # distance et temps cumulés le long du parcours (règle de Tobler sur la pente locale)
        longueur, d_plus = 0.0, 0.0
        d_cum, t_cum = [0.0], [0.0]
        for i in range(1, len(cases)):
            (r1, c1), (r2, c2) = cases[i - 1], cases[i]
            pas = math.hypot((r2 - r1) * f * px, (c2 - c1) * f * px)
            longueur += pas
            d_plus += max(0.0, alt_route[i] - alt_route[i - 1])
            p_moy = (pentes_route[i - 1] + pentes_route[i]) / 2
            v = 6.0 * math.exp(-3.5 * (abs(math.tan(math.radians(p_moy))) + 0.05)) / 3.6
            d_cum.append(longueur)
            t_cum.append(t_cum[-1] + pas / v)

        # cap à suivre en chaque point (azimut vers le point suivant)
        caps = []
        for i in range(len(itineraire)):
            a_pt, b_pt = itineraire[i], itineraire[min(i + 1, len(itineraire) - 1)]
            dx = RAYON_MARS * math.radians(b_pt[0] - a_pt[0]) * math.cos(math.radians((a_pt[1] + b_pt[1]) / 2))
            dy = RAYON_MARS * math.radians(b_pt[1] - a_pt[1])
            caps.append(0.0 if dx == 0 and dy == 0 else (math.degrees(math.atan2(dx, dy)) + 360) % 360)
        if len(caps) > 1:
            caps[-1] = caps[-2]

        pr = np.array(pentes_route)
        t_ref = t_cum[-1] / 60
        t_prud = t_ref / FACTEUR_VITESSE_PRUDENT
        dose = DOSE_RAD_MSV_JOUR * (t_prud / 60) / DUREE_SOL_H
        stats = {"longueur_km": round(longueur / 1000, 2), "denivele_m": round(d_plus), "pente_max": round(float(pr.max()), 1),
                 "prudence_pct": round(float(np.mean(pr >= PENTE_PRUDENCE)) * 100), "temps_ref_min": round(t_ref),
                 "temps_prudent_min": round(t_prud), "dose_msv": round(dose, 3), "dose_jour": DOSE_RAD_MSV_JOUR}
        print(f"Itinéraire : {stats['longueur_km']} km, dénivelé positif {stats['denivele_m']} m, pente max "
              f"{stats['pente_max']}°, {stats['prudence_pct']} % du trajet en zone de prudence")
        print(f"Temps de marche : {stats['temps_ref_min']} min (vitesse de randonnée, règle de Tobler) ; "
              f"{stats['temps_prudent_min']} min en scénario prudent (vitesse x {FACTEUR_VITESSE_PRUDENT})")
        print(f"Radiations : environ {stats['dose_msv']} mSv pendant la marche (scénario prudent), "
              f"sur une base de {DOSE_RAD_MSV_JOUR} mSv/jour mesurés par RAD")
        stats.update({"longueur_m": round(longueur), "alt_min_m": round(min(alt_route), 1),
                      "alt_max_m": round(max(alt_route), 1),
                      "facteur_prudent": FACTEUR_VITESSE_PRUDENT, "duree_sol_h": DUREE_SOL_H})
        with open(f"{SORTIE}/itineraire.geojson", "w", encoding="utf-8") as fh:
            json.dump({"type": "FeatureCollection", "features": [{
                "type": "Feature", "properties": stats,
                "geometry": {"type": "LineString", "coordinates": itineraire}}]}, fh)

        # mission.json : tout ce dont app.html a besoin (les deux écrans lisent ce seul fichier)
        mission = construire_mission({
            "bbox": bbox, "zone_km": ZONE_KM, "itineraire": itineraire, "alt": alt_route,
            "pente": pentes_route, "d_cum": d_cum, "t_cum": t_cum, "caps": caps, "bornes": bornes,
            "habitat": points["habitat"], "habitat_alt": alt_route[0],
            "cibles": points["cibles"], "noms_cibles": noms_cibles,
            "cibles_alt": [alt_route[bornes[k + 1]] for k in range(len(points["cibles"]))],
            "stats": stats, "wpts": wpts, "seuil_rug": seuil_rug, "lissage": meilleur,
        })
        with open(f"{SORTIE}/mission.json", "w", encoding="utf-8") as fh:
            json.dump(mission, fh, ensure_ascii=False, separators=(",", ":"))
        # même contenu en .js : app.html s'ouvre alors d'un simple double-clic (file://)
        with open(f"{SORTIE}/mission.js", "w", encoding="utf-8") as fh:
            fh.write("window.MISSION = " + json.dumps(mission, ensure_ascii=False, separators=(",", ":")) + ";\n")
        print(f"mission.json écrit ({len(itineraire)} points d'itinéraire, {len(mission['etapes'])} étapes)")

    # Visionneuse Leaflet : toutes les couches calées sur les mêmes limites
    bounds = [[bbox[1], bbox[0]], [bbox[3], bbox[2]]]
    html = VIEWER.replace("__BOUNDS__", json.dumps(bounds)) \
                 .replace("__ROUTE__", json.dumps([[lat, lon] for lon, lat in itineraire])) \
                 .replace("__POINTS__", json.dumps(points)) \
                 .replace("__STATS__", json.dumps(stats)) \
                 .replace("__WPTS__", json.dumps([[lat, lon] for lon, lat in wpts]))
    with open(f"{SORTIE}/carte.html", "w", encoding="utf-8") as fh:
        fh.write(html)

    # interface à deux écrans + copie locale de Leaflet, si elles sont à côté du script
    for nom in ("app.html",):
        if os.path.exists(nom):
            shutil.copy(nom, f"{SORTIE}/{nom}")
    if os.path.isdir("leaflet") and not os.path.isdir(f"{SORTIE}/leaflet"):
        shutil.copytree("leaflet", f"{SORTIE}/leaflet")
    cible_finale = "app.html" if os.path.exists(f"{SORTIE}/app.html") else "carte.html"
    print(f"Terminé : ouvrez {SORTIE}/{cible_finale}")

def construire_mission(c):
    """Assemble le fichier mission.json lu par app.html.

    Ne touche à aucun raster : tout arrive déjà calculé dans le dictionnaire `c`,
    ce qui permet de tester cette fonction seule.
    """
    bbox, bornes, stats = c["bbox"], c["bornes"], c["stats"]
    noms = c["noms_cibles"]
    natures = ["crête", "creux"]
    arrivees = list(noms) + ["Habitat"]

    etapes = []
    for k in range(len(bornes) - 1):
        i1, i2 = bornes[k], bornes[k + 1]
        seg = c["pente"][i1:i2 + 1]
        d_pos = sum(max(0.0, c["alt"][i + 1] - c["alt"][i]) for i in range(i1, i2))
        d_neg = sum(min(0.0, c["alt"][i + 1] - c["alt"][i]) for i in range(i1, i2))
        t_ref = (c["t_cum"][i2] - c["t_cum"][i1]) / 60
        depart = "Habitat" if k == 0 else arrivees[k - 1]
        etapes.append({
            "titre": f"{depart} vers {arrivees[k]}", "arrivee": arrivees[k], "i_debut": i1, "i_fin": i2,
            "distance_m": round(c["d_cum"][i2] - c["d_cum"][i1]),
            "denivele_pos_m": round(d_pos), "denivele_neg_m": round(d_neg),
            "pente_max": round(max(seg), 1), "pente_moy": round(sum(seg) / len(seg), 1),
            "temps_ref_min": round(t_ref), "temps_prudent_min": round(t_ref / FACTEUR_VITESSE_PRUDENT),
            "prudence_pct": round(100 * sum(1 for p in seg if p >= PENTE_PRUDENCE) / len(seg)),
        })

    return {
        "nom": "Sortie à pied 01 – boxworks du mont Sharp",
        "site": {"nom": "Mont Sharp (Aeolis Mons), cratère Gale",
                 "centre": [round((bbox[0] + bbox[2]) / 2, 5), round((bbox[1] + bbox[3]) / 2, 5)],
                 "zone_km": c["zone_km"], "corps": "Mars"},
        "bounds": [[bbox[1], bbox[0]], [bbox[3], bbox[2]]],
        "bbox": list(bbox),
        "seuils": {"prudence_deg": PENTE_PRUDENCE, "interdit_deg": PENTE_INTERDITE,
                   "rugosite_interdite_m": round(c["seuil_rug"], 2),
                   "lissage_pente_m": c["lissage"], "pas_grille_m": PAS_ITINERAIRE_M},
        "couches": [
            {"id": "optique", "nom": "Optique HiRISE", "fichier": "optique.jpg", "type": "base", "opacite": 1},
            {"id": "relief", "nom": "Relief ombré (DEM 1 m)", "fichier": "relief.png", "type": "base", "opacite": 1},
            {"id": "pentes", "nom": f"Pentes (0 à 30°)", "fichier": "pentes.png", "type": "overlay", "opacite": 0.55},
            {"id": "dangers", "nom": "Dangers (facile / prudence / interdit)", "fichier": "dangers.png",
             "type": "overlay", "opacite": 0.45, "defaut": True},
            {"id": "cretes_creux", "nom": "Crêtes et creux (boxworks)", "fichier": "cretes_creux.png",
             "type": "overlay", "opacite": 0.8},
        ],
        "habitat": {"nom": "Habitat – rover pressurisé", "lon": c["habitat"][0], "lat": c["habitat"][1],
                    "alt_m": round(c["habitat_alt"], 1), "sol_curiosity": 4544},
        "cibles": [{"nom": noms[i], "lon": p[0], "lat": p[1], "alt_m": round(c["cibles_alt"][i], 1),
                    "nature": natures[i] if i < len(natures) else "cible",
                    "science": SCIENCE_CIBLES[i] if i < len(SCIENCE_CIBLES) else "",
                    "duree_arret_min": DUREE_ARRET_MIN}
                   for i, p in enumerate(c["cibles"])],
        "itineraire": {
            "coordinates": [[round(lon, 7), round(lat, 7)] for lon, lat in c["itineraire"]],
            "altitude_m": [round(v, 2) for v in c["alt"]],
            "pente_deg": [round(v, 1) for v in c["pente"]],
            "distance_cum_m": [round(v, 1) for v in c["d_cum"]],
            "temps_cum_s": [round(v, 1) for v in c["t_cum"]],
            "cap_deg": [round(v, 1) for v in c["caps"]],
        },
        "etapes": etapes,
        "stats": stats,
        "waypoints_curiosity": {"sols": [SOL_MIN, SOL_MAX], "n": len(c["wpts"]),
                                "coordinates": [[lon, lat] for lon, lat in c["wpts"]]},
        "conditions": CONDITIONS,
        "sources": SOURCES,
    }


VIEWER = """<!doctype html><html lang="fr"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Martian Map</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<style>html,body,#map{height:100%;margin:0;background:#1B1512}</style></head><body>
<div id="map"></div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script>
const b = __BOUNDS__, route = __ROUTE__, pts = __POINTS__, wpts = __WPTS__, st = __STATS__;
const map = L.map('map', {crs: L.CRS.EPSG4326, attributionControl: true});
map.fitBounds(b);
const layer = (f, o) => L.imageOverlay(f, b, {opacity: o});
const base = {"Optique (HiRISE)": layer('optique.png', 1), "Relief ombré (DEM 1 m)": layer('relief.png', 1)};
base["Optique (HiRISE)"].addTo(map);
const over = {
  "Pentes": layer('pentes.png', 0.55),
  "Dangers (vert / orange / rouge)": layer('dangers.png', 0.5).addTo(map),
  "Crêtes (orange) et creux (bleu)": layer('cretes_creux.png', 0.8),
  "Trajet de Curiosity (2025)": L.polyline(wpts, {color: '#7FB3D5', weight: 2}),
};
if (route.length) over["Itinéraire le plus rapide et sûr"] = L.polyline(route, {color: '#F6F0E8', weight: 4, dashArray: '8 6'}).addTo(map);
if (pts.habitat) L.marker([pts.habitat[1], pts.habitat[0]]).addTo(map).bindTooltip('Habitat', {permanent: true});
(pts.cibles || []).forEach((c, i) => L.circleMarker([c[1], c[0]], {radius: 9, color: '#E59A6B', weight: 3}).addTo(map).bindTooltip((pts.noms || [])[i] || ('Cible ' + (i + 1)), {permanent: true}));
L.control.layers(base, over, {collapsed: false}).addTo(map);
const info = L.control({position: 'bottomleft'});
info.onAdd = () => { const d = L.DomUtil.create('div'); d.style.cssText = 'background:#1B1512ee;color:#F6F0E8;padding:12px 16px;border-radius:10px;font:14px/1.5 system-ui,sans-serif;max-width:320px';
  d.innerHTML = st.longueur_km ? `<b>Sortie à pied – mont Sharp (boxworks)</b><br>Distance : ${st.longueur_km} km (boucle)<br>Dénivelé positif : ${st.denivele_m} m · pente max ${st.pente_max}°<br>Temps de marche : ${st.temps_ref_min} à ${st.temps_prudent_min} min<br><b>Radiations</b> (RAD, Curiosity) : ${st.dose_jour} mSv/jour en surface<br>Dose pendant la marche : ≈ ${st.dose_msv} mSv` : '<b>Martian Map</b>';
  return d; };
info.addTo(map);
map.attributionControl.addAttribution('Données : NASA/JPL-Caltech/UArizona, USGS, MSL/RAD');
</script></body></html>"""


if __name__ == "__main__":
    main()


In [ ]:
%%writefile app.html
<!doctype html>
<html lang="fr">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Martian Map — planification de sortie à pied</title>
<link rel="stylesheet" href="leaflet/leaflet.css"
      onerror="this.onerror=null;this.href='https://unpkg.com/leaflet@1.9.4/dist/leaflet.css'">
<style>
:root{
  --bg:#1B1512; --bg2:#231B16; --panel:#2A201B; --bord:#3C2E26;
  --txt:#F6F0E8; --muted:#B9A99B; --accent:#E59A6B; --danger:#B5462A;
  --ok:#86B07A; --warn:#E0A33C; --curio:#7FB3D5; --creux:#4C8CE6;
  --r:10px;
}
*{box-sizing:border-box}
html,body{height:100%;margin:0}
body{background:var(--bg);color:var(--txt);
  font:15px/1.45 "Segoe UI",system-ui,-apple-system,Roboto,Helvetica,Arial,sans-serif;
  -webkit-font-smoothing:antialiased;overflow:hidden}
button{font:inherit;color:inherit;cursor:pointer;border:0;background:none}
h1,h2,h3{margin:0;font-weight:600}
.chiffre{font-variant-numeric:tabular-nums}

/* ---------- ossature ---------- */
#app{display:grid;grid-template-rows:auto 1fr;height:100%}
#app[hidden]{display:none}
main{display:grid;grid-template-rows:1fr;min-height:0;position:relative}
main>section{grid-area:1/1;min-height:0}
header{display:flex;align-items:center;gap:18px;padding:10px 16px;
  background:var(--bg2);border-bottom:1px solid var(--bord)}
.brand{font-size:17px;font-weight:700;letter-spacing:.13em}
.brand b{color:var(--accent)}
.brand span{display:block;font-size:11.5px;font-weight:400;letter-spacing:.04em;color:var(--muted)}
nav{display:flex;gap:6px;background:#1B1512;padding:4px;border-radius:var(--r);border:1px solid var(--bord)}
nav button{padding:7px 15px;border-radius:7px;font-size:13.5px;color:var(--muted)}
nav button[aria-selected=true]{background:var(--accent);color:#1B1512;font-weight:600}
.tag{margin-left:auto;font-size:11.5px;color:var(--muted);text-align:right;line-height:1.3}

/* ---------- écran 1 : planification ---------- */
#plan{display:grid;grid-template-columns:1fr 372px;grid-template-rows:1fr auto;
  grid-template-areas:"carte panneau" "conditions panneau";min-height:0}
#map{grid-area:carte;background:#12100E;min-height:0}
#conditions{grid-area:conditions;background:var(--bg2);border-top:1px solid var(--bord);padding:10px 16px}
#panneau{grid-area:panneau;background:var(--bg2);border-left:1px solid var(--bord);
  overflow-y:auto;padding:14px;display:flex;flex-direction:column;gap:12px}

/* contrôle des couches, posé sur la carte */
#couches{position:absolute;top:12px;left:12px;z-index:620;width:252px;
  background:#1B1512ef;border:1px solid var(--bord);border-radius:var(--r);padding:12px 13px;
  backdrop-filter:blur(3px)}
#couches h3{font-size:11px;letter-spacing:.11em;color:var(--muted);text-transform:uppercase;margin-bottom:7px}
#couches h3+h3{margin-top:12px}
#couches label{display:flex;align-items:center;gap:8px;padding:3px 0;font-size:13.5px;cursor:pointer}
#couches input{accent-color:var(--accent);width:15px;height:15px;margin:0;flex:none}
.pastille{width:11px;height:11px;border-radius:3px;flex:none}
.opa{margin-top:10px;display:flex;align-items:center;gap:8px;font-size:12px;color:var(--muted)}
.opa input{flex:1;accent-color:var(--accent)}
#legende{margin-top:11px;padding-top:10px;border-top:1px solid var(--bord);font-size:11.5px;color:var(--muted)}
#legende div{display:flex;align-items:center;gap:7px;padding:1.5px 0}

/* panneau mission */
.carte-info{background:var(--panel);border:1px solid var(--bord);border-radius:var(--r);padding:12px 13px}
.titre-sec{font-size:11px;letter-spacing:.11em;color:var(--muted);text-transform:uppercase;margin-bottom:9px}
.grille{display:grid;grid-template-columns:1fr 1fr;gap:9px 12px}
.kpi .v{font-size:19px;font-weight:650;line-height:1.15}
.kpi .l{font-size:11px;color:var(--muted)}
.kpi .v small{font-size:12px;font-weight:400;color:var(--muted)}
#profil{width:100%;height:84px;display:block;margin-top:4px}
.etape{display:grid;grid-template-columns:26px 1fr;gap:10px;padding:9px;border-radius:9px;
  border:1px solid transparent;cursor:pointer;text-align:left;width:100%}
.etape:hover{background:#ffffff0d}
.etape[aria-current=true]{border-color:var(--accent);background:#E59A6B14}
.puce{width:26px;height:26px;border-radius:50%;display:grid;place-items:center;
  font-size:12px;font-weight:700;background:var(--bord);color:var(--txt)}
.etape[data-type=habitat] .puce{background:var(--txt);color:#1B1512}
.etape[data-type=crête] .puce{background:var(--accent);color:#1B1512}
.etape[data-type=creux] .puce{background:var(--creux);color:#fff}
.etape .nom{font-size:14px;font-weight:600}
.etape .det{font-size:12px;color:var(--muted);margin-top:2px}
.etape .sci{font-size:11.5px;color:var(--accent);margin-top:3px}
.bouton-primaire{width:100%;padding:13px;border-radius:var(--r);background:var(--accent);color:#1B1512;
  font-weight:700;font-size:15px;letter-spacing:.02em}
.bouton-primaire:hover{filter:brightness(1.08)}
.bouton-second{padding:9px 14px;border-radius:9px;border:1px solid var(--bord);color:var(--muted);font-size:13px}
.bouton-second:hover{color:var(--txt);border-color:var(--accent)}

/* conditions */
.cond-tete{display:flex;align-items:baseline;gap:12px;margin-bottom:8px}
.cond-tete h3{font-size:11px;letter-spacing:.11em;color:var(--muted);text-transform:uppercase}
.cond-maj{font-size:11.5px;color:var(--muted)}
#mesures{display:flex;gap:10px;flex-wrap:wrap}
.mesure{flex:1 1 150px;background:var(--panel);border:1px solid var(--bord);border-left:3px solid var(--ok);
  border-radius:8px;padding:7px 10px}
.mesure[data-etat=info]{border-left-color:var(--muted)}
.mesure .n{font-size:10.5px;color:var(--muted);text-transform:uppercase;letter-spacing:.06em}
.mesure .v{font-size:15px;font-weight:650;margin:1px 0}
.mesure .d{font-size:10.5px;color:var(--muted);line-height:1.3}
#sources{margin-top:8px;font-size:11.5px;color:var(--muted)}
#sources summary{cursor:pointer}
#sources ul{margin:6px 0 0;padding-left:18px}

/* ---------- écran 2 : mode marche ---------- */
#marche{display:none;grid-template-rows:auto 1fr auto;height:100%;background:#120F0D}
#marche.actif{display:grid}
.hud-tete{display:flex;align-items:center;gap:16px;padding:14px 26px;border-bottom:1px solid var(--bord)}
.hud-tete .et{font-size:11px;letter-spacing:.14em;color:var(--muted);text-transform:uppercase}
.hud-tete .pt{font-size:26px;font-weight:700;line-height:1.1}
.hud-tete .pt small{display:block;font-size:12.5px;font-weight:400;color:var(--accent);letter-spacing:.02em}
.hud-corps{display:grid;grid-template-columns:repeat(3,minmax(260px,1fr));
  align-items:center;justify-items:center;align-content:center;gap:22px 44px;padding:14px 26px;
  min-height:0;max-width:1240px;width:100%;margin:0 auto}
.hud-corps>*{width:100%;max-width:330px}
#boussole{width:100%;max-width:256px;margin:0 auto;display:block}
.gros{text-align:center}
.gros .v{font-size:84px;font-weight:700;line-height:.95;letter-spacing:-.02em}
.gros .v small{font-size:26px;font-weight:500;color:var(--muted);letter-spacing:0}
.gros .l{font-size:12.5px;letter-spacing:.1em;color:var(--muted);text-transform:uppercase;margin-top:6px}
.gros.sec .v{font-size:46px}
.pile{display:flex;flex-direction:column;gap:14px;justify-content:center}
#alerte{border-radius:12px;padding:14px 18px;border:2px solid var(--ok);background:#86B07A1a}
#alerte .e{font-size:12.5px;letter-spacing:.1em;text-transform:uppercase;color:var(--ok);font-weight:700}
#alerte .p{font-size:40px;font-weight:700;line-height:1.05}
#alerte .t{font-size:12.5px;color:var(--muted);margin-top:2px}
#alerte[data-n=prudence]{border-color:var(--warn);background:#E0A33C1a}
#alerte[data-n=prudence] .e{color:var(--warn)}
#alerte[data-n=limite]{border-color:var(--danger);background:#B5462A26;animation:bat 1.4s infinite}
#alerte[data-n=limite] .e{color:#F08A66}
@keyframes bat{50%{background:#B5462A0d}}
#minicarte{width:100%;max-width:236px;display:block;margin:0 auto;height:auto;background:#1B1512;
  border:1px solid var(--bord);border-radius:12px}
.hud-pied{padding:12px 26px 16px;border-top:1px solid var(--bord)}
.barre{height:7px;border-radius:4px;background:var(--bord);overflow:hidden;margin-bottom:10px}
.barre i{display:block;height:100%;background:var(--accent);width:0}
.cmd{display:flex;align-items:center;gap:12px;flex-wrap:wrap}
.cmd input[type=range]{flex:1;min-width:180px;accent-color:var(--accent)}
.cmd .info{font-size:12.5px;color:var(--muted);min-width:170px}
kbd{font:11px/1 ui-monospace,monospace;border:1px solid var(--bord);border-bottom-width:2px;
  border-radius:4px;padding:3px 5px;color:var(--muted)}

/* divers */
#attente{position:fixed;inset:0;display:grid;place-items:center;background:var(--bg);z-index:900;
  text-align:center;padding:30px}
#attente .boite{max-width:620px}
#attente code{background:var(--panel);padding:2px 7px;border-radius:5px;color:var(--accent);
  font:13px/1.6 ui-monospace,monospace}
.leaflet-container{background:#12100E}
.leaflet-bar,.leaflet-bar a{background:#1B1512;color:var(--txt);border-color:var(--bord)}
.leaflet-bar a:hover{background:var(--panel);color:var(--accent)}
.leaflet-control-attribution{background:#1B1512cc!important;color:var(--muted)!important;font-size:10.5px}
.leaflet-control-attribution a{color:var(--muted)!important}
.etiq{background:#1B1512e8;border:1px solid var(--bord);color:var(--txt);font-size:12px;
  font-weight:600;padding:3px 7px;border-radius:6px;box-shadow:none}
.etiq::before{display:none}
@media (max-width:1100px){
  body{overflow:auto}
  #plan{grid-template-columns:1fr;grid-template-areas:"carte" "panneau" "conditions"}
  #map{height:58vh}
  #panneau{border-left:0;border-top:1px solid var(--bord)}
  .hud-corps{grid-template-columns:1fr}
  .gros .v{font-size:60px}
}
</style>
</head>
<body>

<div id="attente"><div class="boite">
  <h1>Martian Map</h1>
  <p id="attente-txt" style="color:#B9A99B">Chargement des données de mission…</p>
</div></div>

<div id="app" hidden>
  <header>
    <div class="brand">MARTIAN<b>MAP</b>
      <span id="sous-titre">mont Sharp · cratère Gale</span></div>
    <nav role="tablist">
      <button role="tab" data-mode="plan" aria-selected="true">Planification</button>
      <button role="tab" data-mode="marche" aria-selected="false">Mode marche</button>
    </nav>
    <div class="tag">Même lien pour le rover, l'orbite et la Terre<br>
      Données NASA · USGS · MSL (Curiosity)</div>
  </header>

  <main>
    <!-- ======================= ÉCRAN 1 ======================= -->
    <section id="plan">
      <div id="map"></div>

      <div id="couches">
        <h3>Fond de carte</h3>
        <div id="fonds"></div>
        <h3>Couches</h3>
        <div id="surcouches"></div>
        <div class="opa"><span>Opacité</span><input type="range" id="opacite" min="0" max="100" value="100"><span id="opacite-v">100 %</span></div>
        <div id="legende">
          <div><span class="pastille" style="background:#3CB44B"></span>Pente &lt; 10° — marche facile</div>
          <div><span class="pastille" style="background:#E0A33C"></span>10° à 20° — prudence</div>
          <div><span class="pastille" style="background:#B5462A"></span>&gt; 20° ou blocs — interdit</div>
          <div><span class="pastille" style="background:#7FB3D5"></span>Trajet de Curiosity (sols 4534–4830)</div>
        </div>
      </div>

      <aside id="panneau">
        <div class="carte-info">
          <div class="titre-sec">Sortie à pied — bilan</div>
          <div class="grille" id="kpi"></div>
          <svg id="profil" viewBox="0 0 340 84" preserveAspectRatio="none" aria-label="Profil d'altitude"></svg>
          <div style="font-size:11px;color:var(--muted);margin-top:2px">
            Profil d'altitude ; le ruban du bas donne la classe de danger du terrain.
          </div>
        </div>

        <div class="carte-info" style="padding:8px">
          <div class="titre-sec" style="padding:4px 5px 0">Déroulé de la mission</div>
          <div id="etapes"></div>
        </div>

        <button class="bouton-primaire" id="vers-marche">Passer en mode marche →</button>
        <div style="font-size:11.5px;color:var(--muted)">
          Hypothèses réglables : vitesse en combinaison × <b id="h-facteur">0,5</b> de la règle de Tobler ;
          seuils de pente <b id="h-seuils">10° / 20°</b> ; dose RAD <b id="h-dose">0,67 mSv/jour</b>.
        </div>
      </aside>

      <footer id="conditions">
        <div class="cond-tete">
          <h3>Conditions</h3>
          <div class="cond-maj" id="cond-maj"></div>
        </div>
        <div id="mesures"></div>
        <details id="sources"><summary>Sources des données</summary><ul id="liste-sources"></ul></details>
      </footer>
    </section>

    <!-- ======================= ÉCRAN 2 ======================= -->
    <section id="marche">
      <div class="hud-tete">
        <div>
          <div class="et" id="m-etape">Étape 1 sur 3</div>
          <div class="pt" id="m-point">Cible A<small id="m-science"></small></div>
        </div>
        <button class="bouton-second" style="margin-left:auto" id="vers-plan">← Planification</button>
      </div>

      <div class="hud-corps">
        <div>
          <svg id="boussole" viewBox="0 0 200 200" aria-label="Cap à suivre"></svg>
          <div class="gros" style="margin-top:4px">
            <div class="l">Cap à suivre <span id="m-cap-txt"></span></div>
          </div>
        </div>

        <div class="pile">
          <div class="gros"><div class="v chiffre" id="m-dist">0<small> m</small></div>
            <div class="l">jusqu'au prochain point</div></div>
          <div class="gros sec"><div class="v chiffre" id="m-temps">0<small> min</small></div>
            <div class="l">temps estimé jusqu'au point</div></div>
        </div>

        <div class="pile">
          <div id="alerte" data-n="ok">
            <div class="e" id="a-etat">Terrain facile</div>
            <div class="p chiffre" id="a-pente">0,0°</div>
            <div class="t" id="a-txt">pente locale</div>
          </div>
          <div class="gros sec"><div class="v chiffre" id="m-retour">0<small> min</small></div>
            <div class="l">retour à l'habitat</div></div>
          <div><svg id="minicarte" viewBox="0 0 200 200" aria-label="Position sur l'itinéraire"></svg>
            <div class="l" style="font-size:11px;letter-spacing:.08em;color:var(--muted);
              text-transform:uppercase;text-align:center;margin-top:5px">Position sur l'itinéraire</div></div>
        </div>
      </div>

      <div class="hud-pied">
        <div class="barre"><i id="m-barre"></i></div>
        <div class="cmd">
          <button class="bouton-second" id="m-recul">− 25 m</button>
          <button class="bouton-second" id="m-lecture">▶ Lecture</button>
          <button class="bouton-second" id="m-avance">Avancer + 25 m</button>
          <input type="range" id="m-curseur" min="0" max="1000" value="0" aria-label="Position sur l'itinéraire">
          <div class="info chiffre" id="m-info"></div>
          <div class="info" style="min-width:0">
            <kbd>←</kbd> <kbd>→</kbd> avancer · <kbd>espace</kbd> lecture · <kbd>échap</kbd> retour
          </div>
        </div>
      </div>
    </section>
  </main>
</div>

<!-- Données de mission : mission.js permet l'ouverture directe du fichier (file://) ;
     sinon app.html lit mission.json par fetch(). -->
<script src="mission.js" onerror="this.remove()"></script>
<script>
/* ============================================================================
   Martian Map — un seul outil, deux écrans.
   Toutes les données viennent de mission.json (produit par martian_map_pipeline.py).
   ========================================================================= */
const R_MARS = 3396190;
const COUL = {ok:'#86B07A', prudence:'#E0A33C', limite:'#B5462A'};
let M, IT, N, demarre = false;
let posM = 0, lecture = null, mapL = null, couchesL = {}, marqueurPos = null, etapeActive = -1;

const fr = (x, d = 0) => x.toLocaleString('fr-FR', {minimumFractionDigits: d, maximumFractionDigits: d});
const el = id => document.getElementById(id);

/* --- géométrie Mars (équirectangulaire locale) --- */
const rad = Math.PI / 180;
function metres(a, b) {
  const dx = R_MARS * (b[0] - a[0]) * rad * Math.cos((a[1] + b[1]) / 2 * rad);
  const dy = R_MARS * (b[1] - a[1]) * rad;
  return Math.hypot(dx, dy);
}
function capDeg(a, b) {
  const dx = R_MARS * (b[0] - a[0]) * rad * Math.cos((a[1] + b[1]) / 2 * rad);
  const dy = R_MARS * (b[1] - a[1]) * rad;
  return (Math.atan2(dx, dy) / rad + 360) % 360;
}
const ROSE = ['N','NNE','NE','ENE','E','ESE','SE','SSE','S','SSO','SO','OSO','O','ONO','NO','NNO'];
const rose = c => ROSE[Math.round(c / 22.5) % 16];

/* --- interpolation le long de l'itinéraire, par distance parcourue --- */
function indexDist(s) {
  const D = IT.distance_cum_m;
  let lo = 0, hi = D.length - 1;
  while (hi - lo > 1) { const m = (lo + hi) >> 1; (D[m] <= s) ? lo = m : hi = m; }
  return lo;
}
function etat(s) {
  const D = IT.distance_cum_m, C = IT.coordinates;
  s = Math.max(0, Math.min(s, D[D.length - 1]));
  const i = indexDist(s), j = Math.min(i + 1, D.length - 1);
  const f = D[j] > D[i] ? (s - D[i]) / (D[j] - D[i]) : 0;
  const mix = (A) => A[i] + (A[j] - A[i]) * f;
  return {
    s, i,
    lon: C[i][0] + (C[j][0] - C[i][0]) * f,
    lat: C[i][1] + (C[j][1] - C[i][1]) * f,
    alt: mix(IT.altitude_m),
    pente: mix(IT.pente_deg),
    t: mix(IT.temps_cum_s)            // secondes, vitesse de référence (Tobler)
  };
}
/* cap : on suit l'itinéraire validé, pas la ligne droite vers la cible */
function capSuivi(s) {
  const e = etat(s), f = etat(Math.min(s + 20, IT.distance_cum_m[IT.distance_cum_m.length - 1] - 0.1));
  return metres([e.lon, e.lat], [f.lon, f.lat]) < 0.5 ? IT.cap_deg[e.i] : capDeg([e.lon, e.lat], [f.lon, f.lat]);
}
const minutes = s => s / 60 / M.stats.facteur_prudent;   // temps prudent (hypothèse combinaison)
function niveau(p) {
  if (p >= M.seuils.interdit_deg - 3) return 'limite';
  if (p >= M.seuils.prudence_deg) return 'prudence';
  return 'ok';
}

/* ========================== ÉCRAN 1 : CARTE ============================= */
function construireCarte() {
  const b = M.bounds;
  mapL = L.map('map', {crs: L.CRS.EPSG4326, attributionControl: true, zoomControl: false, maxZoom: 24});
  L.control.zoom({position: 'bottomright'}).addTo(mapL);
  mapL.fitBounds(b);
  mapL.attributionControl.addAttribution(
    'NASA/JPL-Caltech · USGS Astrogeology · MSL (Curiosity)');

  const fonds = el('fonds'), surc = el('surcouches');
  const ligne = (type, id, nom, actif, pastille) => {
    const l = document.createElement('label');
    l.innerHTML = `<input type="${type === 'base' ? 'radio' : 'checkbox'}"
        ${type === 'base' ? 'name="fond"' : ''} ${actif ? 'checked' : ''} data-id="${id}">
      ${pastille ? `<span class="pastille" style="background:${pastille}"></span>` : ''}
      <span>${nom}</span>`;
    (type === 'base' ? fonds : surc).appendChild(l);
    return l.querySelector('input');
  };

  // couches raster déclarées dans mission.json
  M.couches.forEach((c, k) => {
    const actif = c.type === 'base' ? k === 0 : !!c.defaut;
    const o = L.imageOverlay(c.fichier, b, {opacity: c.opacite, className: 'c-' + c.id});
    couchesL[c.id] = {obj: o, conf: c};
    if (actif) o.addTo(mapL);
    ligne(c.type, c.id, c.nom, actif).onchange = e => basculer(c.id, e.target.checked, c.type);
  });

  // couches vectorielles
  const route = IT.coordinates.map(p => [p[1], p[0]]);
  couchesL.itineraire = {obj: L.layerGroup([
    L.polyline(route, {color: '#1B1512', weight: 8, opacity: .55}),
    L.polyline(route, {color: '#F6F0E8', weight: 4, dashArray: '9 6'})
  ]), conf: {}};
  couchesL.curiosity = {obj: L.polyline(
    M.waypoints_curiosity.coordinates.map(p => [p[1], p[0]]),
    {color: '#7FB3D5', weight: 2, opacity: .9}), conf: {}};
  couchesL.itineraire.obj.addTo(mapL);
  ligne('over', 'itineraire', 'Itinéraire le plus rapide et sûr', true, '#F6F0E8')
    .onchange = e => basculer('itineraire', e.target.checked);
  ligne('over', 'curiosity', `Trajet de Curiosity (${M.waypoints_curiosity.n} positions)`, false, '#7FB3D5')
    .onchange = e => basculer('curiosity', e.target.checked);

  // points de mission
  const pin = (p, nom, coul, remplissage) => L.circleMarker([p.lat, p.lon], {
      radius: 9, color: coul, weight: 3, fillColor: remplissage, fillOpacity: 1})
    .addTo(mapL).bindTooltip(nom, {permanent: true, direction: 'right', className: 'etiq', offset: [10, 0]});
  pin(M.habitat, 'Habitat', '#F6F0E8', '#1B1512');
  M.cibles.forEach(c => pin(c, c.nom, c.nature === 'creux' ? '#4C8CE6' : '#E59A6B', '#1B1512'));

  marqueurPos = L.circleMarker([M.habitat.lat, M.habitat.lon],
    {radius: 7, color: '#1B1512', weight: 2, fillColor: '#E59A6B', fillOpacity: 1});

  el('opacite').oninput = e => {
    const v = +e.target.value / 100;
    el('opacite-v').textContent = Math.round(v * 100) + ' %';
    M.couches.forEach(c => { if (c.type !== 'base' && mapL.hasLayer(couchesL[c.id].obj))
      couchesL[c.id].obj.setOpacity(c.opacite * v); });
  };
}
function basculer(id, actif, type) {
  const c = couchesL[id];
  if (type === 'base') {   // un seul fond à la fois
    M.couches.filter(x => x.type === 'base').forEach(x => mapL.removeLayer(couchesL[x.id].obj));
    c.obj.addTo(mapL).bringToBack();
  } else if (actif) {
    c.obj.addTo(mapL);
    if (c.conf.opacite) c.obj.setOpacity(c.conf.opacite * (+el('opacite').value / 100));
  } else mapL.removeLayer(c.obj);
  if (mapL.hasLayer(couchesL.itineraire.obj)) couchesL.itineraire.obj.eachLayer(l => l.bringToFront());
}

/* ===================== ÉCRAN 1 : PANNEAU MISSION ======================== */
function construirePanneau() {
  const s = M.stats;
  const kpi = [
    [fr(s.longueur_km, 2) + ' <small>km</small>', 'distance (boucle)'],
    [fr(s.temps_prudent_min) + ' <small>min de marche</small>', `${fr(s.temps_ref_min)} min à vitesse de randonnée`],
    ['+' + fr(s.denivele_m) + ' <small>m</small>', 'dénivelé positif'],
    [fr(s.pente_max, 1) + '<small>°</small>', `pente maximale · ${fr(s.prudence_pct)} % en prudence`],
    ['≈ ' + fr(s.dose_msv, 3) + ' <small>mSv</small>', 'dose reçue pendant la marche'],
    [fr(M.cibles.length) + ' <small>cibles</small>', 'une crête + un creux (boxworks)']
  ];
  el('kpi').innerHTML = kpi.map(([v, l]) =>
    `<div class="kpi"><div class="v chiffre">${v}</div><div class="l">${l}</div></div>`).join('');

  el('h-facteur').textContent = fr(M.stats.facteur_prudent, 1);
  el('h-seuils').textContent = `${fr(M.seuils.prudence_deg)}° / ${fr(M.seuils.interdit_deg)}°`;
  el('h-dose').textContent = fr(M.stats.dose_jour, 2) + ' mSv/jour';
  el('sous-titre').textContent = `${M.site.nom} · zone ${fr(M.site.zone_km, 0)} × ${fr(M.site.zone_km, 0)} km`;

  // étapes
  const cible = nom => M.cibles.find(c => c.nom === nom);
  const rangs = [{type: 'habitat', nom: M.habitat.nom, det:
      `Départ · ${fr(M.habitat.alt_m, 0)} m d'altitude · position réelle de Curiosity au sol ${M.habitat.sol_curiosity}`,
      sci: '', i: 0, puce: '◆'}];
  M.etapes.forEach((e, k) => {
    const c = cible(e.arrivee);
    rangs.push({
      type: c ? c.nature : 'habitat', nom: e.arrivee, puce: k + 1, i: k + 1, etape: e,
      det: `${fr(e.distance_m)} m · ${fr(e.temps_prudent_min)} min · D+ ${fr(e.denivele_pos_m)} m · pente max ${fr(e.pente_max, 1)}°`,
      sci: c ? `${c.science} · arrêt ${fr(c.duree_arret_min)} min` : 'Fin de sortie — remontée dans le rover pressurisé'
    });
  });
  el('etapes').innerHTML = rangs.map((r, k) => `
    <button class="etape" data-type="${r.type}" data-k="${k}">
      <span class="puce">${r.puce}</span>
      <span><span class="nom">${r.nom}</span>
        <span class="det">${r.det}</span>
        ${r.sci ? `<span class="sci">${r.sci}</span>` : ''}</span>
    </button>`).join('');
  el('etapes').querySelectorAll('.etape').forEach(b => b.onclick = () => viserEtape(+b.dataset.k));

  // conditions
  el('cond-maj').textContent = M.conditions.maj;
  el('mesures').innerHTML = M.conditions.mesures.map(m =>
    `<div class="mesure" data-etat="${m.etat}"><div class="n">${m.nom}</div>
      <div class="v chiffre">${m.valeur}</div><div class="d">${m.detail}</div></div>`).join('');
  el('liste-sources').innerHTML = M.sources.map(s => `<li>${s}</li>`).join('');

  profil();
}

function viserEtape(k) {
  etapeActive = k;
  el('etapes').querySelectorAll('.etape').forEach(b =>
    b.setAttribute('aria-current', +b.dataset.k === k));
  if (k === 0) { mapL.setView([M.habitat.lat, M.habitat.lon], mapL.getZoom()); }
  else {
    const e = M.etapes[k - 1];
    const seg = IT.coordinates.slice(e.i_debut, e.i_fin + 1).map(p => [p[1], p[0]]);
    mapL.fitBounds(L.latLngBounds(seg).pad(0.25));
  }
  profil();
}

/* profil d'altitude + ruban de danger */
function profil(curseur = null) {
  const svg = el('profil'), W = 340, H = 84, hb = 8;
  const D = IT.distance_cum_m, A = IT.altitude_m, P = IT.pente_deg;
  const dmax = D[D.length - 1], amin = Math.min(...A), amax = Math.max(...A);
  const x = d => d / dmax * W, y = a => 4 + (1 - (a - amin) / (amax - amin || 1)) * (H - hb - 10);
  let ligne = '', ruban = '';
  D.forEach((d, i) => ligne += `${i ? 'L' : 'M'}${x(d).toFixed(1)} ${y(A[i]).toFixed(1)}`);
  for (let i = 0; i < D.length - 1; i++)
    ruban += `<rect x="${x(D[i]).toFixed(1)}" y="${H - hb}" width="${Math.max(x(D[i + 1]) - x(D[i]), .8).toFixed(1)}"
       height="${hb}" fill="${COUL[niveau(P[i])]}"/>`;
  let marques = '';
  (etapeActive > 0 ? [M.etapes[etapeActive - 1]] : M.etapes).forEach(e => {
    if (etapeActive > 0) marques += `<rect x="${x(D[e.i_debut]).toFixed(1)}" y="0"
      width="${(x(D[e.i_fin]) - x(D[e.i_debut])).toFixed(1)}" height="${H}" fill="#E59A6B1f"/>`;
    marques += `<line x1="${x(D[e.i_fin]).toFixed(1)}" y1="0" x2="${x(D[e.i_fin]).toFixed(1)}" y2="${H - hb}"
      stroke="#B9A99B" stroke-width=".8" stroke-dasharray="3 3"/>`;
  });
  const cur = curseur === null ? '' :
    `<line x1="${x(curseur).toFixed(1)}" y1="0" x2="${x(curseur).toFixed(1)}" y2="${H - hb}"
       stroke="#E59A6B" stroke-width="2"/>`;
  svg.innerHTML = `${marques}
    <path d="${ligne}L${W} ${H - hb}L0 ${H - hb}Z" fill="#E59A6B1a"/>
    <path d="${ligne}" fill="none" stroke="#E59A6B" stroke-width="1.6"/>
    ${ruban}${cur}
    <text x="2" y="11" fill="#B9A99B" font-size="9">${fr(amax, 0)} m</text>
    <text x="2" y="${H - hb - 2}" fill="#B9A99B" font-size="9">${fr(amin, 0)} m</text>
    <text x="${W - 2}" y="${H - hb - 2}" fill="#B9A99B" font-size="9" text-anchor="end">${fr(dmax, 0)} m</text>`;
}

/* ======================= ÉCRAN 2 : MODE MARCHE ========================== */
function construireHud() {
  el('m-curseur').max = Math.round(IT.distance_cum_m[IT.distance_cum_m.length - 1]);
  el('m-avance').onclick = () => deplacer(posM + 25);
  el('m-recul').onclick = () => deplacer(posM - 25);
  el('m-curseur').oninput = e => deplacer(+e.target.value);
  el('m-lecture').onclick = basculerLecture;
}
function basculerLecture() {
  if (lecture) { clearInterval(lecture); lecture = null; el('m-lecture').textContent = '▶ Lecture'; return; }
  el('m-lecture').textContent = '❚❚ Pause';
  lecture = setInterval(() => {
    const fin = IT.distance_cum_m[IT.distance_cum_m.length - 1];
    if (posM >= fin) return basculerLecture();
    deplacer(posM + 8);
  }, 120);
}
function deplacer(s) {
  const fin = IT.distance_cum_m[IT.distance_cum_m.length - 1];
  posM = Math.max(0, Math.min(s, fin));
  majHud();
}
function majHud() {
  const e = etat(posM), fin = IT.distance_cum_m[IT.distance_cum_m.length - 1];
  const D = IT.distance_cum_m;

  // étape courante = première étape dont l'arrivée est devant nous
  let k = M.etapes.findIndex(x => D[x.i_fin] > posM + 0.01);
  if (k < 0) k = M.etapes.length - 1;
  const et = M.etapes[k], cibleEt = M.cibles.find(c => c.nom === et.arrivee);

  el('m-etape').textContent = `Étape ${k + 1} sur ${M.etapes.length}`;
  el('m-point').innerHTML = `${et.arrivee}<small>${cibleEt ? cibleEt.science : 'Retour au rover pressurisé'}</small>`;

  const dRest = Math.max(0, D[et.i_fin] - posM);
  const tRest = Math.max(0, minutes(IT.temps_cum_s[et.i_fin] - e.t));
  el('m-dist').innerHTML = `${fr(dRest)}<small> m</small>`;
  el('m-temps').innerHTML = `${fr(Math.max(tRest, dRest > 0 ? 1 : 0))}<small> min</small>`;

  // retour à l'habitat : marche restante + arrêts scientifiques restants
  const arretsRestants = M.cibles
    .filter(c => { const ee = M.etapes.find(x => x.arrivee === c.nom); return ee && D[ee.i_fin] > posM; })
    .reduce((a, c) => a + c.duree_arret_min, 0);
  const tHab = minutes(IT.temps_cum_s[IT.temps_cum_s.length - 1] - e.t) + arretsRestants;
  el('m-retour').innerHTML = `${fr(tHab)}<small> min</small>`;

  // cap
  const cap = capSuivi(posM);
  el('m-cap-txt').textContent = `${fr(cap)}° ${rose(cap)}`;
  boussole(cap, e.pente);

  // alerte de pente
  const n = niveau(e.pente);
  el('alerte').dataset.n = n;
  el('a-etat').textContent = n === 'ok' ? 'Terrain facile'
    : n === 'prudence' ? 'Prudence' : 'Pente forte';
  el('a-pente').textContent = fr(e.pente, 1) + '°';
  el('a-txt').textContent = n === 'ok' ? `pente locale · seuil de prudence ${fr(M.seuils.prudence_deg)}°`
    : n === 'prudence' ? `pente locale · au-delà de ${fr(M.seuils.prudence_deg)}°, pas lents`
    : `pente locale · limite de sécurité ${fr(M.seuils.interdit_deg)}° — cherchez un appui`;

  // progression, dose, position
  el('m-barre').style.width = (posM / fin * 100) + '%';
  el('m-curseur').value = Math.round(posM);
  const hEcoul = minutes(e.t) / 60 + (M.cibles.reduce((a, c) => a + c.duree_arret_min, 0) - arretsRestants) / 60;
  el('m-info').textContent = `${fr(posM)} / ${fr(fin)} m · ${fr(e.alt, 0)} m d'altitude · `
    + `dose ≈ ${fr(M.stats.dose_jour * hEcoul / M.stats.duree_sol_h, 3)} mSv`;
  minicarte(e, cap);
  if (mapL) {
    marqueurPos.setLatLng([e.lat, e.lon]);
    if (!mapL.hasLayer(marqueurPos)) marqueurPos.addTo(mapL);
    profil(posM);
  }
}

function boussole(cap, pente) {
  const C = COUL[niveau(pente)];
  let graduations = '';
  for (let a = 0; a < 360; a += 15) {
    const g = a % 90 === 0, r1 = g ? 72 : 78, x1 = 100 + r1 * Math.sin(a * rad), y1 = 100 - r1 * Math.cos(a * rad),
      x2 = 100 + 86 * Math.sin(a * rad), y2 = 100 - 86 * Math.cos(a * rad);
    graduations += `<line x1="${x1.toFixed(1)}" y1="${y1.toFixed(1)}" x2="${x2.toFixed(1)}" y2="${y2.toFixed(1)}"
      stroke="#3C2E26" stroke-width="${g ? 2.5 : 1.2}"/>`;
  }
  const lettres = [['N', 0], ['E', 90], ['S', 180], ['O', 270]].map(([t, a]) => {
    const x = 100 + 60 * Math.sin(a * rad), y = 100 - 60 * Math.cos(a * rad);
    return `<text x="${x.toFixed(1)}" y="${(y + 4).toFixed(1)}" fill="#B9A99B" font-size="13"
      text-anchor="middle" font-weight="600">${t}</text>`;
  }).join('');
  el('boussole').innerHTML = `
    <circle cx="100" cy="100" r="92" fill="none" stroke="#3C2E26" stroke-width="1.5"/>
    ${graduations}${lettres}
    <g transform="rotate(${cap.toFixed(1)} 100 100)">
      <path d="M100 20 L119 110 L100 96 L81 110 Z" fill="${C}"/>
    </g>
    <circle cx="100" cy="100" r="7" fill="#1B1512" stroke="${C}" stroke-width="2.5"/>`;
}

let CADRE = null;
function cadreItineraire() {                       // carre englobant l'itineraire, avec marge
  const C = IT.coordinates;
  let x0 = Infinity, x1 = -Infinity, y0 = Infinity, y1 = -Infinity;
  C.concat([[M.habitat.lon, M.habitat.lat]]).forEach(p => {
    x0 = Math.min(x0, p[0]); x1 = Math.max(x1, p[0]);
    y0 = Math.min(y0, p[1]); y1 = Math.max(y1, p[1]);
  });
  const cx = (x0 + x1) / 2, cy = (y0 + y1) / 2;
  const d = Math.max(x1 - x0, (y1 - y0) / Math.cos(cy * rad)) * 1.18 / 2;
  return [cx - d, cy - d * Math.cos(cy * rad), cx + d, cy + d * Math.cos(cy * rad)];
}
function minicarte(e, cap) {
  const bb = CADRE || (CADRE = cadreItineraire()), W = 200, pad = 12;
  const x = lon => pad + (lon - bb[0]) / (bb[2] - bb[0]) * (W - 2 * pad);
  const y = lat => pad + (bb[3] - lat) / (bb[3] - bb[1]) * (W - 2 * pad);
  const d = IT.coordinates.map((p, i) => `${i ? 'L' : 'M'}${x(p[0]).toFixed(1)} ${y(p[1]).toFixed(1)}`).join('');
  const wp = M.waypoints_curiosity.coordinates
    .map((p, i) => `${i ? 'L' : 'M'}${x(p[0]).toFixed(1)} ${y(p[1]).toFixed(1)}`).join('');
  const cibles = M.cibles.map(c => `<circle cx="${x(c.lon).toFixed(1)}" cy="${y(c.lat).toFixed(1)}" r="4"
      fill="none" stroke="${c.nature === 'creux' ? '#4C8CE6' : '#E59A6B'}" stroke-width="2"/>`).join('');
  el('minicarte').innerHTML = `
    <path d="${wp}" fill="none" stroke="#7FB3D5" stroke-width="1" opacity=".5"/>
    <path d="${d}" fill="none" stroke="#F6F0E8" stroke-width="1.6" opacity=".85"/>
    ${cibles}
    <rect x="${(x(M.habitat.lon) - 4).toFixed(1)}" y="${(y(M.habitat.lat) - 4).toFixed(1)}" width="8" height="8"
      fill="none" stroke="#F6F0E8" stroke-width="2"/>
    <g transform="translate(${x(e.lon).toFixed(1)} ${y(e.lat).toFixed(1)}) rotate(${cap.toFixed(1)})">
      <path d="M0 -8 L5 6 L0 2 L-5 6 Z" fill="${COUL[niveau(e.pente)]}"/></g>
    <text x="${W - 4}" y="${W - 5}" fill="#B9A99B" font-size="8" text-anchor="end">${fr(M.stats.longueur_km, 2)} km</text>`;
}

/* ============================== MODES =================================== */
function mode(m) {
  document.querySelectorAll('nav button').forEach(b =>
    b.setAttribute('aria-selected', b.dataset.mode === m));
  el('plan').style.display = m === 'plan' ? 'grid' : 'none';
  el('marche').classList.toggle('actif', m === 'marche');
  if (m === 'plan' && mapL) setTimeout(() => mapL.invalidateSize(), 30);
  if (m === 'marche') majHud();
  else if (lecture) basculerLecture();
}

/* ============================== DÉMARRAGE =============================== */
function lancer(mission) {
  M = mission; IT = M.itineraire;
  el('attente').remove();
  el('app').hidden = false;
  construireCarte();
  construirePanneau();
  construireHud();
  deplacer(0);
  mode('plan');
  const recadrer = () => { mapL.invalidateSize(); mapL.fitBounds(M.bounds); };
  requestAnimationFrame(() => requestAnimationFrame(recadrer));
  window.addEventListener('resize', () => { clearTimeout(window.__t); window.__t = setTimeout(recadrer, 200); });
  document.querySelectorAll('nav button').forEach(b => b.onclick = () => mode(b.dataset.mode));
  el('vers-marche').onclick = () => mode('marche');
  el('vers-plan').onclick = () => mode('plan');
  document.addEventListener('keydown', ev => {
    if (!el('marche').classList.contains('actif')) return;
    if (ev.key === 'ArrowRight') { deplacer(posM + 25); ev.preventDefault(); }
    if (ev.key === 'ArrowLeft') { deplacer(posM - 25); ev.preventDefault(); }
    if (ev.key === ' ') { basculerLecture(); ev.preventDefault(); }
    if (ev.key === 'Escape') mode('plan');
  });
}
function echec(msg) {
  const a = el('attente');
  if (!a) return;
  a.querySelector('.boite').innerHTML = `<h1>Martian Map</h1>
    <p style="color:#E0A33C">${msg}</p>
    <p style="color:#B9A99B">Servez le dossier par HTTP, puis rouvrez cette page :</p>
    <p><code>python -m http.server 8000</code></p>
    <p style="color:#B9A99B">puis <code>http://localhost:8000/app.html</code></p>`;
}
function donnees() {
  if (window.MISSION) return lancer(window.MISSION);     // mission.js : marche aussi en file://
  fetch('mission.json').then(r => r.ok ? r.json() : Promise.reject(r.status))
    .then(lancer)
    .catch(e => echec(`Impossible de lire <code>mission.json</code> (${e}).`));
}
/* Leaflet : copie locale si elle existe, sinon unpkg (besoin du réseau) */
function chargerLeaflet(fait) {
  if (window.L) return fait();
  const s = document.createElement('script');
  s.src = 'leaflet/leaflet.js';
  s.onload = fait;
  s.onerror = () => {
    const u = document.createElement('script');
    u.src = 'https://unpkg.com/leaflet@1.9.4/dist/leaflet.js';
    u.onload = fait;
    u.onerror = () => echec('Leaflet introuvable (ni copie locale, ni réseau).');
    document.head.appendChild(u);
  };
  document.head.appendChild(s);
}
chargerLeaflet(donnees);
</script>
</body>
</html>


In [ ]:
# Copie locale de Leaflet : app.html s'ouvre ensuite sans connexion internet.
!mkdir -p leaflet/images
!wget -q -O leaflet/leaflet.js  https://unpkg.com/leaflet@1.9.4/dist/leaflet.js
!wget -q -O leaflet/leaflet.css https://unpkg.com/leaflet@1.9.4/dist/leaflet.css
for f in ['marker-icon.png', 'marker-icon-2x.png', 'marker-shadow.png']:
    !wget -q -O leaflet/images/{f} https://unpkg.com/leaflet@1.9.4/dist/images/{f}
!ls -l leaflet leaflet/images


## Étape 3 – Vérifier l'accès aux données USGS (quelques secondes)
Si les deux lignes affichent une taille et un système de coordonnées, tout est bon.

In [ ]:
import rasterio
for url in ["/vsicurl/https://planetarymaps.usgs.gov/mosaic/Mars/MSL/MSL_Gale_DEM_Mosaic_1m_v3.tif",
            "/vsicurl/https://asc-pds-services.s3.us-west-2.amazonaws.com/mosaic/Mars/MSL/MSL_Gale_HiRISE-LRGB_78quads_sharp_cog.tif"]:
    with rasterio.open(url) as s:
        print(url.split('/')[-1], s.width, 'x', s.height, '| pixel', round(abs(s.transform.a), 2), 'm |', s.crs.to_string()[:60], '| tuilé :', s.profile.get('tiled'))

## Étape 4 – Lancer l'extraction et le calcul (quelques minutes)
À la fin, notez les lignes **Validation** et **Itinéraire** : ce sont vos chiffres pour le pitch.

In [ ]:
!python martian_map_pipeline.py

## Étape 5 – Voir un aperçu ici

In [ ]:
from IPython.display import Image, display
for f in ['optique.jpg', 'relief.png', 'pentes.png', 'dangers.png', 'cretes_creux.png']:
    print(f); display(Image('sortie/' + f, width=500))


## Étape 6 – Télécharger le dossier résultat
Dézippez-le, puis ouvrez **`app.html`** (interface à deux écrans). `carte.html` reste disponible comme visionneuse simple. Tous les fichiers doivent rester dans le même dossier.


In [ ]:
!zip -r -q sortie.zip sortie
from google.colab import files
files.download('sortie.zip')